# AEFS Sensitivity Analysis & Runtime Benchmark

This notebook runs sensitivity analysis for AEFS hyperparameters and measures runtime.

**What it does:**
1. SHAP threshold sensitivity (1%, 2%, 3%, 5%)
2. Stability threshold sensitivity (20%, 30%, 40%, 50%)
3. Runtime comparison (AEFS vs baselines)
4. Convergence funnel (features at each pipeline stage)
5. Uploads all results + figures to HuggingFace

**Corrections applied to this notebook:**
* The AEEEM loader now builds `defective` from the `class` ground-truth column
  (`buggy`/`clean`). It previously derived it from the `numberOfBugsFoundUntil:`
  *feature*, which is a prior-bug-history metric, not the label -- that produced
  meaningless AEEEM sensitivity/runtime/convergence numbers.
* The experiments now run the **full four-stage** AEFS pipeline
  (RUP -> SFS -> SHAP -> stability) with SMOTE on source projects only, so the
  numbers are comparable with `results/*_stage4_full_results.csv`.
* Data/output paths fall back to the repository's `data/` and
  `sensitivity/{results,figures}` folders when not running on Kaggle.
* `regen_sensitivity.py` in the repository root implements the identical
  experiments as plain Python (used to regenerate the committed CSVs).

**Setup on Kaggle:**
1. Upload `AEFS_data.zip` as dataset (same as main notebook)
2. Add HF_TOKEN to Secrets (same as main notebook)
3. Run all cells


## 0. Setup

In [ ]:
!pip install -q lightgbm shap imbalanced-learn tqdm matplotlib seaborn python-dotenv huggingface_hub

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='shap')

import os
import re
import time
import math
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations
from dataclasses import dataclass, field

from scipy.stats import pointbiserialr
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif, RFE
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef, f1_score, roc_auc_score
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE

# shap is optional: if its optional numba dependency cannot be loaded in this
# environment, fall back to LightGBM's pred_contrib, which returns the exact
# same TreeSHAP values for tree ensembles.
try:
    import shap
    HAVE_SHAP = True
except Exception as _shap_err:
    shap = None
    HAVE_SHAP = False
    print(f'shap unavailable ({_shap_err}); using LightGBM pred_contrib (exact TreeSHAP)')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)

RANDOM_STATE = 42
print('Setup complete. shap backend:', 'shap.TreeExplainer' if HAVE_SHAP else 'pred_contrib')


In [ ]:
# Data paths -- Kaggle first, then this repository's local data/ folder
KAGGLE_DATA_ROOTS = [
    Path('/kaggle/input/datasets/mihjihad/aefs-data'),
    Path('/kaggle/input/datasets/afrojahaqueshornaly/aefs-data'),
]
LOCAL_DATA_ROOTS = [Path.cwd() / 'data', Path.cwd().parent / 'data']


def _resolve_data_dir(name):
    tried = []
    for root in KAGGLE_DATA_ROOTS + LOCAL_DATA_ROOTS:
        d = root / name
        tried.append(str(d))
        if d.exists():
            return d
    raise FileNotFoundError(f'{name} not found; tried: {tried}')


RAW_TERA_DIR = _resolve_data_dir('tera')
RAW_AEEEM_DIR = _resolve_data_dir('aeeem')

# Output dirs: Kaggle writes to /kaggle/working, locally to sensitivity/
if Path('/kaggle/working').exists():
    RESULTS_DIR = '/kaggle/working/results'
    FIG_DIR = '/kaggle/working/figures'
else:
    RESULTS_DIR = str(Path.cwd() / 'sensitivity' / 'results')
    FIG_DIR = str(Path.cwd() / 'sensitivity' / 'figures')

Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
Path(FIG_DIR).mkdir(parents=True, exist_ok=True)

print('TeraPromise files:', len(list(RAW_TERA_DIR.glob('*.csv'))))
print('AEEEM files:', len(list(RAW_AEEEM_DIR.glob('*.csv'))))
print('RESULTS_DIR:', RESULTS_DIR)
print('FIG_DIR:', FIG_DIR)


## 1. Data Loading

In [ ]:
def load_aeeem():
    """AEEEM loader.

    The defect label is the `class` column (`buggy` -> 1, `clean` -> 0).
    The bug-count columns (e.g. `numberOfBugsFoundUntil:`) are *features*,
    not labels -- they record cumulative bugs across earlier versions and
    must never be used to derive `defective`.
    """
    pool = {}
    for f in sorted(RAW_AEEEM_DIR.glob('*.csv')):
        df = pd.read_csv(f)
        name = f.stem

        df = df.drop(columns=[c for c in df.columns if c.strip().lower() == 'id'],
                     errors='ignore')
        df = df.rename(columns={c: re.sub(r'[^A-Za-z0-9_]', '', c) for c in df.columns})

        if 'class' not in df.columns:
            raise ValueError(f'{f.name}: expected label column "class", got {list(df.columns)}')

        feature_cols = [c for c in df.columns if c != 'class']
        for c in feature_cols:
            df[c] = pd.to_numeric(df[c], errors='coerce')
        df['class'] = df['class'].astype(str).str.strip().str.lower().map({'buggy': 1, 'clean': 0})
        df = df.dropna(axis=0, how='any')
        df['defective'] = df['class'].astype(int)
        df = df.drop(columns=['class'])
        df = df.drop_duplicates().reset_index(drop=True)

        if len(df) >= 20 and df['defective'].sum() >= 2 and df['defective'].sum() < len(df):
            pool[name] = df
    return pool


def load_tera():
    """TeraPromise loader: `bug` is a count, binarized to `defective`."""
    pool = {}
    for f in sorted(RAW_TERA_DIR.glob('*.csv')):
        df = pd.read_csv(f)
        name = f.stem
        df = df.drop(columns=[c for c in df.columns if c.strip().lower()
                              in {'name', 'name.1', 'version'}], errors='ignore')
        if 'bug' not in df.columns:
            raise ValueError(f'{f.name}: expected label column "bug"')
        feature_cols = [c for c in df.columns if c != 'bug']
        for c in feature_cols:
            df[c] = pd.to_numeric(df[c], errors='coerce')
        df['bug'] = pd.to_numeric(df['bug'], errors='coerce')
        df = df.dropna(axis=0, how='any')
        df['defective'] = (df['bug'] > 0).astype(int)
        df = df.drop(columns=['bug'])
        df = df.drop_duplicates().reset_index(drop=True)
        if len(df) >= 20 and df['defective'].sum() >= 2 and df['defective'].sum() < len(df):
            pool[name] = df
    return pool


aeeem_pool = load_aeeem()
tera_pool = load_tera()
print(f'AEEEM: {len(aeeem_pool)} projects -> {sorted(aeeem_pool)}')
print(f'TeraPromise: {len(tera_pool)} projects')

# Validate all pool entries have 'defective' column
for pool_name, pool in [('AEEEM', aeeem_pool), ('TeraPromise', tera_pool)]:
    for proj, df in pool.items():
        assert 'defective' in df.columns, f'{pool_name}/{proj}: missing defective column'
        assert set(df['defective'].unique()) <= {0, 1}, f'{pool_name}/{proj}: non-binary label'
print('All pool entries validated: defective column present and binary.')


## 2. Pipeline Functions

In [ ]:
# LightGBM params for SFS/search (lightweight)
# NOTE: random_state is deliberately *not* baked in here -- callers pass it
# explicitly (LGBMClassifier(**LGBM_SFS_PARAMS, random_state=...)).
LGBM_SFS_PARAMS = dict(
    n_estimators=60, max_depth=5, num_leaves=15,
    verbosity=-1, n_jobs=-1,
)

# LightGBM params for final model
LGBM_FINAL_PARAMS = dict(
    n_estimators=300, max_depth=-1, num_leaves=31, learning_rate=0.05,
    verbosity=-1, n_jobs=-1, random_state=RANDOM_STATE,
)

RF_FINAL_PARAMS = dict(
    n_estimators=300, max_depth=None, n_jobs=-1, random_state=RANDOM_STATE,
)


In [ ]:
def tree_shap_values(clf, X):
    """SHAP values for a fitted tree model.

    Uses shap.TreeExplainer when shap is importable, otherwise LightGBM's
    own pred_contrib -- both compute exact TreeSHAP and agree to machine
    precision for LightGBM models.
    """
    if HAVE_SHAP:
        sv = shap.TreeExplainer(clf).shap_values(X)
        if isinstance(sv, list):
            sv = sv[1]
        if sv.ndim == 3:
            sv = sv[:, :, 1]
        return np.asarray(sv)
    return np.asarray(clf.predict(X, pred_contrib=True))[:, :-1]


def rup_filter(df, variance_thr=1e-4, relevance_thr=0.02, redundancy_thr=0.90, min_features=5):
    """Stage 2A: Rapid Uninformative Pruning (variance -> relevance -> redundancy)."""
    cols = [c for c in df.columns if c != 'defective']
    X = df[cols]
    y = df['defective']

    # Pass 1: variance
    vt = VarianceThreshold(threshold=variance_thr)
    vt.fit(X)
    kept = [c for c, s in zip(cols, vt.get_support()) if s]

    # Pass 2: relevance (point-biserial correlation with the label)
    relevance = {}
    for c in kept:
        try:
            corr, _ = pointbiserialr(y, X[c])
        except Exception:
            corr = 0.0
        relevance[c] = abs(corr) if not np.isnan(corr) else 0.0
    relevant = [c for c in kept if relevance[c] >= relevance_thr]

    # Pass 3: redundancy -- among correlated pairs keep the more relevant one
    if len(relevant) > 1:
        corr_matrix = X[relevant].corr().abs()
        ordered = sorted(relevant, key=lambda c: relevance[c], reverse=True)
        survivors, removed = [], set()
        for feat in ordered:
            if feat in removed:
                continue
            survivors.append(feat)
            for other in ordered:
                if other == feat or other in removed or other in survivors:
                    continue
                if corr_matrix.loc[feat, other] >= redundancy_thr:
                    removed.add(other)
        final = survivors
    else:
        final = list(relevant)

    if len(final) < min_features:
        ranked = sorted((c for c in kept if c not in final),
                        key=lambda c: relevance[c], reverse=True)
        final = final + ranked[:min_features - len(final)]
    return final


def _cv_mcc(X, y, k=3, random_state=RANDOM_STATE):
    """Mean MCC of a lightweight LightGBM under stratified k-fold CV."""
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=random_state)
    scores = []
    for tr, va in skf.split(X, y):
        clf = LGBMClassifier(**LGBM_SFS_PARAMS, random_state=random_state)
        clf.fit(X.iloc[tr], y.iloc[tr])
        scores.append(matthews_corrcoef(y.iloc[va], clf.predict(X.iloc[va])))
    return float(np.mean(scores))


def sequential_forward_selection(X, y, candidates, cv_folds=3, min_improvement=0.001,
                                 min_features=5, random_state=RANDOM_STATE):
    """Stage 2B: greedy forward selection scored by cross-validated MCC."""
    remaining, selected = list(candidates), []
    best_score = -1.0
    while remaining:
        scores = {f: _cv_mcc(X[selected + [f]], y, k=cv_folds, random_state=random_state)
                  for f in remaining}
        best_feat = max(scores, key=scores.get)
        if scores[best_feat] - best_score < min_improvement and selected:
            break
        selected.append(best_feat)
        remaining.remove(best_feat)
        best_score = scores[best_feat]

    if len(selected) < min_features and remaining:
        scores_fb = {f: _cv_mcc(X[selected + [f]], y, k=cv_folds, random_state=random_state)
                     for f in remaining}
        while len(selected) < min_features and remaining:
            best_fb = max(scores_fb, key=scores_fb.get)
            selected.append(best_fb)
            remaining.remove(best_fb)
            scores_fb.pop(best_fb)
            for f in list(scores_fb):
                scores_fb[f] = _cv_mcc(X[selected + [f]], y, k=cv_folds, random_state=random_state)
    return selected


def shap_refine(X, y, candidates, threshold_frac, min_features=5):
    """Stage 2C: SHAP-based refinement with configurable contribution threshold."""
    if len(candidates) <= min_features:
        return list(candidates)

    clf = LGBMClassifier(**LGBM_SFS_PARAMS, random_state=RANDOM_STATE)
    clf.fit(X[candidates], y)

    mean_abs = np.abs(tree_shap_values(clf, X[candidates])).mean(axis=0)
    importance = pd.Series(mean_abs, index=candidates)
    total = importance.sum()
    if total == 0:
        return list(candidates)

    frac = importance / total
    keep = frac[frac >= threshold_frac].index.tolist()
    if len(keep) < min_features:
        keep = importance.head(min_features).index.tolist()
    return keep


def _jaccard(a, b):
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)


def stability_check(X, y, project_labels, candidates, threshold, min_features=5):
    """Stage 2D: cross-project stability check with configurable threshold."""
    if len(candidates) <= min_features:
        return list(candidates), 1.0

    top_k = max(1, math.ceil(0.7 * len(candidates)))
    project_top_sets = {}
    for proj in project_labels.unique():
        mask = project_labels == proj
        Xp = X.loc[mask, candidates]
        yp = y.loc[mask]
        if len(Xp) < 20 or yp.value_counts().min() < 2:
            continue
        clf = LGBMClassifier(**LGBM_SFS_PARAMS, random_state=RANDOM_STATE)
        clf.fit(Xp, yp)
        mean_abs = np.abs(tree_shap_values(clf, Xp)).mean(axis=0)
        ranked = pd.Series(mean_abs, index=candidates).sort_values(ascending=False)
        project_top_sets[proj] = set(ranked.head(top_k).index)

    if not project_top_sets:
        return list(candidates), 1.0

    n_eval = len(project_top_sets)
    presence = {f: sum(f in s for s in project_top_sets.values()) / n_eval
                for f in candidates}
    selected = [f for f in candidates if presence[f] >= threshold]
    if len(selected) < min_features:
        selected = sorted(candidates, key=lambda f: presence[f], reverse=True)[:min_features]

    proj_names = list(project_top_sets)
    if len(proj_names) >= 2:
        mean_jac = float(np.mean([_jaccard(project_top_sets[a], project_top_sets[b])
                                  for a, b in combinations(proj_names, 2)]))
    else:
        mean_jac = 1.0
    return selected, mean_jac


In [ ]:
def apply_smote(X, y, random_state=RANDOM_STATE):
    """SMOTE on source data only; no-op when a fold is too small."""
    minority = y.value_counts().min()
    if minority < 2:
        return X, y
    return SMOTE(random_state=random_state,
                 k_neighbors=min(5, minority - 1)).fit_resample(X, y)


def lopo_folds(pool, min_test_defective=1):
    """Leave-One-Project-Out folds.

    Per fold: MinMax scaling, Stage 2A RUP and SMOTE are all fit on the
    SOURCE projects only -- never on the held-out target project.
    """
    names = list(pool.keys())
    for target in names:
        test_df = pool[target]
        source_names = [n for n in names if n != target]
        source_dfs = [pool[n] for n in source_names]
        train_df = pd.concat(source_dfs, axis=0, ignore_index=True)

        if test_df['defective'].sum() < min_test_defective:
            continue

        project_labels = pd.Series(
            np.concatenate([[n] * len(df) for n, df in zip(source_names, source_dfs)]),
            index=train_df.index,
        )

        feature_cols = [c for c in train_df.columns if c != 'defective']
        X_raw, y_raw = train_df[feature_cols], train_df['defective']
        X_te_raw, y_te = test_df[feature_cols], test_df['defective']

        scaler = MinMaxScaler()
        X_sc = pd.DataFrame(scaler.fit_transform(X_raw), columns=feature_cols, index=X_raw.index)
        X_te = pd.DataFrame(scaler.transform(X_te_raw), columns=feature_cols, index=X_te_raw.index)

        candidates = rup_filter(pd.concat([X_sc, y_raw], axis=1))

        X_all, X_te_all = X_sc.copy(), X_te.copy()
        X_sc, X_te = X_sc[candidates], X_te[candidates]
        X_presmote, y_presmote = X_sc.copy(), y_raw.copy()
        X_train, y_train = apply_smote(X_sc, y_raw)

        yield {
            'target': target,
            'X_train': X_train, 'y_train': y_train,
            'X_test': X_te, 'y_test': y_te,
            'X_all': X_all, 'X_te_all': X_te_all,
            'X_presmote': X_presmote, 'y_presmote': y_presmote,
            'project_labels': project_labels,
            'rup_candidates': candidates,
        }


def evaluate_features(fold, features):
    """Train the final LightGBM on `features` and score the held-out target."""
    lgbm = LGBMClassifier(**LGBM_FINAL_PARAMS)
    lgbm.fit(fold['X_train'][features], fold['y_train'])
    proba = lgbm.predict_proba(fold['X_test'][features])[:, 1]
    pred = (proba >= 0.5).astype(int)
    try:
        auc = roc_auc_score(fold['y_test'], proba)
    except ValueError:
        auc = float('nan')
    return dict(
        n_features=len(features),
        mcc=round(matthews_corrcoef(fold['y_test'], pred), 4),
        f1=round(f1_score(fold['y_test'], pred, zero_division=0), 4),
        auc=round(auc, 4),
    )


## 3. SHAP Threshold Sensitivity

In [ ]:
def run_shap_sensitivity(pool, pool_name, shap_thresholds=None):
    """Stage 2C contribution-threshold sweep (Stage 2D held at 0.30)."""
    if shap_thresholds is None:
        shap_thresholds = [0.01, 0.02, 0.03, 0.05]

    rows = []
    folds = list(lopo_folds(pool))
    print(f'[{pool_name}] SHAP sensitivity: {len(folds)} folds x {len(shap_thresholds)} thresholds')

    for i, fold in enumerate(folds, 1):
        sfs_feats = sequential_forward_selection(fold['X_train'], fold['y_train'],
                                                 fold['rup_candidates'])
        for thr in shap_thresholds:
            shap_feats = shap_refine(fold['X_train'], fold['y_train'], sfs_feats, thr)
            final_feats, _ = stability_check(fold['X_presmote'], fold['y_presmote'],
                                             fold['project_labels'], shap_feats, 0.30)
            m = evaluate_features(fold, final_feats)
            rows.append(dict(target=fold['target'], shap_threshold=thr, **m))

        if i % 10 == 0 or i == len(folds):
            print(f'  Fold {i}/{len(folds)}: {fold["target"]} done')

    df = pd.DataFrame(rows)
    df.to_csv(f'{RESULTS_DIR}/{pool_name}_sensitivity_shap.csv', index=False)
    print(f'[{pool_name}] SHAP sensitivity: saved {len(df)} rows')
    return df


In [ ]:
shap_aeeem = run_shap_sensitivity(aeeem_pool, 'aeeem')
print()
shap_aeeem.groupby('shap_threshold')[['mcc', 'f1', 'n_features']].mean().round(4)

In [ ]:
shap_tera = run_shap_sensitivity(tera_pool, 'tera')
print()
shap_tera.groupby('shap_threshold')[['mcc', 'f1', 'n_features']].mean().round(4)

## 4. Stability Threshold Sensitivity

In [ ]:
def run_stability_sensitivity(pool, pool_name, stability_thresholds=None):
    """Stage 2D threshold sweep (Stage 2C held at 0.02)."""
    if stability_thresholds is None:
        stability_thresholds = [0.20, 0.30, 0.40, 0.50]

    rows = []
    folds = list(lopo_folds(pool))
    print(f'[{pool_name}] Stability sensitivity: {len(folds)} folds x {len(stability_thresholds)} thresholds')

    for i, fold in enumerate(folds, 1):
        sfs_feats = sequential_forward_selection(fold['X_train'], fold['y_train'],
                                                 fold['rup_candidates'])
        shap_feats = shap_refine(fold['X_train'], fold['y_train'], sfs_feats, 0.02)

        for thr in stability_thresholds:
            final_feats, jac = stability_check(fold['X_presmote'], fold['y_presmote'],
                                               fold['project_labels'], shap_feats, thr)
            m = evaluate_features(fold, final_feats)
            rows.append(dict(target=fold['target'], stability_threshold=thr,
                             mean_jaccard=round(jac, 4), **m))

        if i % 10 == 0 or i == len(folds):
            print(f'  Fold {i}/{len(folds)}: {fold["target"]} done')

    df = pd.DataFrame(rows)
    df.to_csv(f'{RESULTS_DIR}/{pool_name}_sensitivity_stability.csv', index=False)
    print(f'[{pool_name}] Stability sensitivity: saved {len(df)} rows')
    return df


In [ ]:
stab_aeeem = run_stability_sensitivity(aeeem_pool, 'aeeem')
print()
stab_aeeem.groupby('stability_threshold')[['mcc', 'f1', 'n_features']].mean().round(4)

In [ ]:
stab_tera = run_stability_sensitivity(tera_pool, 'tera')
print()
stab_tera.groupby('stability_threshold')[['mcc', 'f1', 'n_features']].mean().round(4)

## 5. Runtime Comparison

In [ ]:
def measure_runtime(pool, pool_name, n_folds=None):
    """Per-fold wall time for AEFS vs the three selection baselines.

    Baseline training sets are built outside the timed region so the clock
    measures feature selection + model fit + predict only. Baselines start
    from the same pre-RUP feature space AEFS starts from, and k is matched
    to AEFS's own final feature count for that fold.
    """
    methods = ['AEFS', 'AllFeatures', 'FilterMI', 'WrapperRFE']
    timings = {m: [] for m in methods}

    folds = list(lopo_folds(pool))
    if n_folds:
        folds = folds[:n_folds]
    print(f'[{pool_name}] Runtime measurement: {len(folds)} folds')

    for i, fold in enumerate(folds, 1):
        X_all_sm, y_all_sm = apply_smote(fold['X_all'], fold['y_presmote'])
        X_te_all = fold['X_te_all']

        # --- AEFS (full four-stage pipeline) ---
        t0 = time.perf_counter()
        candidates = rup_filter(pd.concat([fold['X_all'], fold['y_presmote']], axis=1))
        sfs_feats = sequential_forward_selection(fold['X_train'], fold['y_train'], candidates)
        shap_feats = shap_refine(fold['X_train'], fold['y_train'], sfs_feats, 0.02)
        final_feats, _ = stability_check(fold['X_presmote'], fold['y_presmote'],
                                         fold['project_labels'], shap_feats, 0.30)
        k = len(final_feats)
        lgbm = LGBMClassifier(**LGBM_FINAL_PARAMS)
        lgbm.fit(fold['X_train'][final_feats], fold['y_train'])
        lgbm.predict(fold['X_test'][final_feats])
        timings['AEFS'].append(time.perf_counter() - t0)

        # --- AllFeatures ---
        t0 = time.perf_counter()
        lgbm = LGBMClassifier(**LGBM_FINAL_PARAMS)
        lgbm.fit(X_all_sm, y_all_sm)
        lgbm.predict(X_te_all)
        timings['AllFeatures'].append(time.perf_counter() - t0)

        # --- FilterMI ---
        t0 = time.perf_counter()
        selector = SelectKBest(score_func=lambda X_, y_: mutual_info_classif(
            X_, y_, random_state=RANDOM_STATE), k=min(k, X_all_sm.shape[1]))
        selector.fit(X_all_sm, y_all_sm)
        feats = list(X_all_sm.columns[selector.get_support()])
        lgbm = LGBMClassifier(**LGBM_FINAL_PARAMS)
        lgbm.fit(X_all_sm[feats], y_all_sm)
        lgbm.predict(X_te_all[feats])
        timings['FilterMI'].append(time.perf_counter() - t0)

        # --- WrapperRFE ---
        t0 = time.perf_counter()
        rfe = RFE(LGBMClassifier(**LGBM_SFS_PARAMS, random_state=RANDOM_STATE),
                  n_features_to_select=min(k, X_all_sm.shape[1]), step=0.2)
        rfe.fit(X_all_sm, y_all_sm)
        feats = list(X_all_sm.columns[rfe.support_])
        lgbm = LGBMClassifier(**LGBM_FINAL_PARAMS)
        lgbm.fit(X_all_sm[feats], y_all_sm)
        lgbm.predict(X_te_all[feats])
        timings['WrapperRFE'].append(time.perf_counter() - t0)

        if i % 5 == 0 or i == len(folds):
            print(f'  Fold {i}/{len(folds)}: {fold["target"]} done')

    rows = [dict(method=m, mean_time_s=round(float(np.mean(t)), 2),
                 std_time_s=round(float(np.std(t)), 2),
                 total_time_s=round(float(np.sum(t)), 2))
            for m, t in timings.items()]
    df = pd.DataFrame(rows)
    df.to_csv(f'{RESULTS_DIR}/{pool_name}_runtime.csv', index=False)
    print(f'[{pool_name}] Runtime: saved')
    return df


In [ ]:
runtime_aeeem = measure_runtime(aeeem_pool, 'aeeem')
print()
runtime_aeeem.to_string(index=False)

In [ ]:
runtime_tera = measure_runtime(tera_pool, 'tera', n_folds=10)
print()
runtime_tera.to_string(index=False)

## 6. Convergence Funnel

In [ ]:
def measure_convergence(pool, pool_name):
    """Stage-wise feature count: original -> RUP -> SFS -> SHAP -> stability."""
    rows = []
    folds = list(lopo_folds(pool))
    print(f'[{pool_name}] Convergence funnel: {len(folds)} folds')

    for i, fold in enumerate(folds, 1):
        original = fold['X_all'].shape[1]
        candidates = rup_filter(pd.concat([fold['X_all'], fold['y_presmote']], axis=1))
        sfs_feats = sequential_forward_selection(fold['X_train'], fold['y_train'], candidates)
        shap_feats = shap_refine(fold['X_train'], fold['y_train'], sfs_feats, 0.02)
        final_feats, _ = stability_check(fold['X_presmote'], fold['y_presmote'],
                                         fold['project_labels'], shap_feats, 0.30)

        rows.append(dict(target=fold['target'], original=original,
                         after_rup=len(candidates), after_sfs=len(sfs_feats),
                         after_shap=len(shap_feats), after_stability=len(final_feats)))

        if i % 10 == 0 or i == len(folds):
            print(f'  Fold {i}/{len(folds)}: {fold["target"]} done')

    df = pd.DataFrame(rows)
    df.to_csv(f'{RESULTS_DIR}/{pool_name}_convergence.csv', index=False)
    print(f'[{pool_name}] Convergence: saved')
    return df


In [ ]:
conv_aeeem = measure_convergence(aeeem_pool, 'aeeem')
print()
print('AEEEM mean:')
print(conv_aeeem.mean(numeric_only=True).round(1))

In [ ]:
conv_tera = measure_convergence(tera_pool, 'tera')
print()
print('TeraPromise mean:')
print(conv_tera.mean(numeric_only=True).round(1))

## 7. Generate Figures

In [ ]:
def save_fig(name, fig):
    fig.savefig(os.path.join(FIG_DIR, f'{name}.pdf'), bbox_inches='tight')
    fig.savefig(os.path.join(FIG_DIR, f'{name}.png'), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved: {name}')

In [ ]:
# SHAP sensitivity plot
fig, ax = plt.subplots(figsize=(8, 5))
summary = shap_aeeem.groupby('shap_threshold')['mcc'].mean().reset_index()
bars = ax.bar(summary['shap_threshold'].astype(str), summary['mcc'],
              color='#457b9d', edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, summary['mcc']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax.set_xlabel('SHAP Contribution Threshold')
ax.set_ylabel('Mean MCC')
ax.set_title('SHAP Threshold Sensitivity (AEEEM, LightGBM)')
save_fig('sensitivity_shap_aeeem', fig)

In [ ]:
# Stability sensitivity plot
fig, ax = plt.subplots(figsize=(8, 5))
summary = stab_aeeem.groupby('stability_threshold')['mcc'].mean().reset_index()
bars = ax.bar(summary['stability_threshold'].astype(str), summary['mcc'],
              color='#e76f51', edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, summary['mcc']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax.set_xlabel('Stability Threshold (% source agreement)')
ax.set_ylabel('Mean MCC')
ax.set_title('Stability Threshold Sensitivity (AEEEM, LightGBM)')
save_fig('sensitivity_stability_aeeem', fig)

In [ ]:
# Convergence funnel
fig, ax = plt.subplots(figsize=(8, 5))
stages = ['Original', 'After RUP', 'After SFS', 'After SHAP', 'Final']
aeeem_means = [conv_aeeem['original'].mean(), conv_aeeem['after_rup'].mean(),
               conv_aeeem['after_sfs'].mean(), conv_aeeem['after_shap'].mean(),
               conv_aeeem['after_stability'].mean()]
tera_means = [conv_tera['original'].mean(), conv_tera['after_rup'].mean(),
              conv_tera['after_sfs'].mean(), conv_tera['after_shap'].mean(),
              conv_tera['after_stability'].mean()]

x = np.arange(len(stages))
width = 0.35
bars1 = ax.bar(x - width/2, aeeem_means, width, label='AEEEM', color='#457b9d', edgecolor='black')
bars2 = ax.bar(x + width/2, tera_means, width, label='TeraPromise', color='#2a9d8f', edgecolor='black')

for bar, val in zip(bars1, aeeem_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar, val in zip(bars2, tera_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('Average Number of Features')
ax.set_title('Pipeline Convergence Funnel')
ax.set_xticks(x)
ax.set_xticklabels(stages)
ax.legend()
save_fig('convergence_funnel', fig)


In [ ]:
# Runtime comparison
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(runtime_aeeem['method'], runtime_aeeem['mean_time_s'],
              yerr=runtime_aeeem['std_time_s'], color=['#e63946', '#2a9d8f', '#e9c46a', '#264653'],
              edgecolor='black', capsize=5)
for bar, val in zip(bars, runtime_aeeem['mean_time_s']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}s', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Mean Runtime per Fold (s)')
ax.set_title('Runtime Comparison (AEEEM, LightGBM)')
save_fig('runtime_comparison', fig)

In [ ]:
# List all generated figures
print('\nAll generated figures:')
for f in sorted(os.listdir(FIG_DIR)):
    size_kb = os.path.getsize(os.path.join(FIG_DIR, f)) / 1024
    print(f'  {f:45s} {size_kb:7.1f} KB')

## 8. Summary Tables

In [ ]:
print('='*60)
print('SHAP Sensitivity (AEEEM)')
print('='*60)
print(shap_aeeem.groupby('shap_threshold')[['mcc', 'f1', 'n_features']].mean().round(4))
print()

print('='*60)
print('Stability Sensitivity (AEEEM)')
print('='*60)
print(stab_aeeem.groupby('stability_threshold')[['mcc', 'f1', 'n_features']].mean().round(4))
print()

print('='*60)
print('Runtime (AEEEM)')
print('='*60)
print(runtime_aeeem.to_string(index=False))
print()

print('='*60)
print('Convergence (AEEEM)')
print('='*60)
print(conv_aeeem.mean(numeric_only=True).round(1))

## 9. Upload to HuggingFace

In [ ]:
from huggingface_hub import HfApi, login

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('Loaded HF_TOKEN from Kaggle Secrets')
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')
    if HF_TOKEN:
        print('Loaded HF_TOKEN from .env file')

if not HF_TOKEN:
    raise RuntimeError(
        'HF_TOKEN not found. On Kaggle: add it via Add Data > Add Environment Variable. '
        'Locally: put HF_TOKEN=hf_... in your .env file.')

login(token=HF_TOKEN)
api = HfApi()

HF_REPO = 'MoshinAli/aefs-cpdp-results'

try:
    api.create_repo(HF_REPO, repo_type='dataset', private=True, exist_ok=True)
    print(f'Repo ready: https://huggingface.co/datasets/{HF_REPO}')
except Exception as e:
    print(f'Repo setup: {e}')

In [ ]:
upload_files = []

# Result CSVs
for f in os.listdir(RESULTS_DIR):
    if f.endswith('.csv'):
        upload_files.append(os.path.join(RESULTS_DIR, f))

# Figures
for f in os.listdir(FIG_DIR):
    if f.endswith(('.pdf', '.png')):
        upload_files.append(os.path.join(FIG_DIR, f))

print(f'Uploading {len(upload_files)} files to {HF_REPO}:\n')
for fp in upload_files:
    fname = os.path.basename(fp)
    if fp.endswith('.csv'):
        hf_path = f'sensitivity/results/{fname}'
    elif fp.endswith(('.pdf', '.png')):
        hf_path = f'sensitivity/figures/{fname}'
    else:
        hf_path = f'sensitivity/{fname}'
    try:
        api.upload_file(
            path_or_fileobj=fp,
            path_in_repo=hf_path,
            repo_id=HF_REPO,
            repo_type='dataset',
        )
        print(f'  OK: {hf_path}')
    except Exception as e:
        print(f'  FAIL: {hf_path} -- {e}')

print(f'\nDone. Browse: https://huggingface.co/datasets/{HF_REPO}/tree/main')

## 10. Update TikZ Figures (for paper)

After getting real results, copy these coordinates into `draft/figures/sensitivity_*.tikz` and `draft/figures/convergence_funnel.tikz`.

In [ ]:
# Print TikZ-ready coordinates
print('% === Copy into sensitivity_shap.tikz ===')
shap_summary = shap_aeeem.groupby('shap_threshold')['mcc'].mean()
for thr, mcc in shap_summary.items():
    print(f'% ({thr:.0%}, {mcc:.4f})')

print()
print('% === Copy into sensitivity_stability.tikz ===')
stab_summary = stab_aeeem.groupby('stability_threshold')['mcc'].mean()
for thr, mcc in stab_summary.items():
    print(f'% ({thr:.0%}, {mcc:.4f})')

print()
print('% === Copy into convergence_funnel.tikz ===')
print(f'% AEEEM: {conv_aeeem["original"].mean():.1f} -> {conv_aeeem["after_rup"].mean():.1f} '
      f'-> {conv_aeeem["after_sfs"].mean():.1f} -> {conv_aeeem["after_shap"].mean():.1f} '
      f'-> {conv_aeeem["after_stability"].mean():.1f}')
print(f'% Tera:  {conv_tera["original"].mean():.1f} -> {conv_tera["after_rup"].mean():.1f} '
      f'-> {conv_tera["after_sfs"].mean():.1f} -> {conv_tera["after_shap"].mean():.1f} '
      f'-> {conv_tera["after_stability"].mean():.1f}')
